In [1]:
import os
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# ==========================================
# 1. 配置与初始化 (假设你已启动 Neo4j 数据库)
# ==========================================
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "your_password"

# 假设使用本地/开源LLM或OpenAI接口
os.environ["OPENAI_API_KEY"] = "your_api_key"
# os.environ["OPENAI_API_BASE"] = "http://localhost:8000/v1" # 如果用本地vLLM/Ollama

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ==========================================
# 2. Neo4j 图谱构建 (对应感知层与规划层的图谱存储)
# ==========================================
class SceneGraphBuilder:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def build_graph_from_perception(self, perception_data):
        """
        将感知层输出的结构化数据写入Neo4j，构建以拓扑节点为索引的场景图谱
        perception_data 示例: 
        [
            {"topo_node": "Node_1", "room": "Kitchen", "objects": ["fridge", "cup"]},
            {"topo_node": "Node_2", "room": "LivingRoom", "objects": ["sofa", "table", "cup"]},
            {"topo_node": "Node_3", "room": "Kitchen", "objects": ["stove"]}
        ]
        """
        with self.driver.session() as session:
            # 清空旧数据 (仅演示用)
            session.run("MATCH (n) DETACH DELETE n")
            
            for data in perception_data:
                # 创建拓扑节点
                session.run("""
                    MERGE (t:TopoNode {id: $topo_node})
                    SET t.room = $room
                """, topo_node=data["topo_node"], room=data["room"])
                
                # 创建物体节点并建立关系 (物体 IN 拓扑节点)
                for obj in data["objects"]:
                    session.run("""
                        MATCH (t:TopoNode {id: $topo_node})
                        MERGE (o:Object {name: $obj})
                        MERGE (o)-[:LOCATED_IN]->(t)
                    """, topo_node=data["topo_node"], obj=obj)
            
            # 建立拓扑节点之间的连接关系 (模拟 Voronoi + A* 的邻接图)
            session.run("""
                MATCH (t1:TopoNode {id: 'Node_1'}), (t2:TopoNode {id: 'Node_2'})
                MERGE (t1)-[:CONNECTED_TO {cost: 1}]->(t2)
            """)
            session.run("""
                MATCH (t1:TopoNode {id: 'Node_1'}), (t2:TopoNode {id: 'Node_3'})
                MERGE (t1)-[:CONNECTED_TO {cost: 1}]->(t2)
            """)

# ==========================================
# 3. Graph RAG 检索器 (对应规划层的 Context 增强)
# ==========================================
class GraphRAGRetriever:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def retrieve_context(self, query_objects):
        """
        根据指令中涉及的物体，检索相关的拓扑节点和空间关系，作为 RAG 的 Context
        """
        context = []
        with self.driver.session() as session:
            for obj in query_objects:
                # 查询物体所在的拓扑节点及房间
                result = session.run("""
                    MATCH (o:Object {name: $obj})-[:LOCATED_IN]->(t:TopoNode)
                    RETURN t.id AS topo_id, t.room AS room
                """, obj=obj)
                
                for record in result:
                    context.append(f"Object '{obj}' is located in Topo Node '{record['topo_id']}' (Room: {record['room']}).")
                    
                    # 进一步检索该拓扑节点的邻居 (为 A* 规划提供备选路径)
                    neighbors = session.run("""
                        MATCH (t:TopoNode {id: $topo_id})-[:CONNECTED_TO]->(neighbor:TopoNode)
                        RETURN neighbor.id AS neighbor_id, neighbor.room AS neighbor_room
                    """, topo_id=record['topo_id'])
                    
                    for nb in neighbors:
                        context.append(f"Node '{record['topo_id']}' is connected to Node '{nb['neighbor_id']}' (Room: {nb['neighbor_room']}).")
                        
        return "\n".join(context)

# ==========================================
# 4. LLM 规划 Agent (对应决策层与规划层)
# ==========================================
def agent_plan_with_graph_rag(user_command: str):
    # 1. 简单的实体提取 (实际项目中用 LLM 提取)
    # 假设指令: "Go to the cup in the kitchen"
    query_objects = ["cup"] 
    
    # 2. Graph RAG 检索
    retriever = GraphRAGRetriever(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    graph_context = retriever.retrieve_context(query_objects)
    
    print(f"=== Retrieval Context ===\n{graph_context}\n=========================")
    
    # 3. 构建 RAG Prompt (结合图谱上下文约束 LLM 输出)
    system_prompt = """You are a robot navigation planner. 
Based on the user command and the Graph Context retrieved from the knowledge graph, 
determine the best target topological node for the robot.
Output a JSON with: {"target_node": "Node_X", "reason": "..."}."""

    human_prompt = f"""User Command: {user_command}
    
Graph Context:
{graph_context}

Which topological node should the robot go to?"""

    # 4. LLM 推理
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt)
    ])
    
    return response.content

# ==========================================
# 5. 主程序运行流
# ==========================================
if __name__ == "__main__":
    # 模拟感知层输入数据 (YOLO + GroundingDINO 提取结果)
    mock_perception_data = [
        {"topo_node": "Node_1", "room": "Kitchen", "objects": ["fridge", "cup"]},
        {"topo_node": "Node_2", "room": "LivingRoom", "objects": ["sofa", "table", "cup"]},
        {"topo_node": "Node_3", "room": "Kitchen", "objects": ["stove"]}
    ]

    # 初始化并构建图谱
    graph_builder = SceneGraphBuilder(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    graph_builder.build_graph_from_perception(mock_perception_data)
    graph_builder.close()

    # 模拟高层自然语言指令
    command = "Go to the cup in the kitchen."
    
    # 执行 GraphRAG 增强的决策
    llm_decision = agent_plan_with_graph_rag(command)
    
    print(f"User Command: {command}")
    print(f"LLM Decision (Output to Execution Layer):\n{llm_decision}")


/home/chendawww/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('127.0.0.1:7687',)):
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)